[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kasparvonbeelen/contracts/blob/main/influence_explore.ipynb)

In [ ]:
!git clone https://github.com/kasparvonbeelen/contracts.git


In [ ]:
%cd contracts

In [ ]:
!pip install -q -e .

In [ ]:
from tools.influence_analysis_helpers import visualize_sentence_similarity_timeline
import pandas as pd
import numpy as np
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, clear_output

# Make Plotly + widgets work consistently in Colab.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

pio.renderers.default = "colab" if IN_COLAB else "notebook_connected"

if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

In [ ]:
clause_type = "modification"  # privacy, liability, termination, indemnity, warranty
data_path = "results/tous"
influence_scores = pd.read_csv(f"{data_path}/influence_scores_{clause_type}.tsv" , sep="\t")
nodes_df = pd.read_csv(f"{data_path}/nodes_{clause_type}.tsv" , sep="\t")
emb_norm = np.loadtxt(f'{data_path}/emb_norm_{clause_type}.txt', delimiter=',')

In [ ]:
# Get top influential sentence IDs.
top_ids = influence_scores.nlargest(1000, "influence_score")["node_id"].astype(int).tolist()

if "node_id" in nodes_df.columns:
    valid_sentence_ids = sorted(nodes_df["node_id"].astype(int).unique().tolist())
else:
    valid_sentence_ids = list(range(len(nodes_df)))

default_sentence_id = top_ids[0] if top_ids else valid_sentence_ids[0]
if default_sentence_id not in valid_sentence_ids:
    default_sentence_id = valid_sentence_ids[0]

sentence_id_slider = widgets.IntSlider(
    value=int(default_sentence_id),
    min=int(valid_sentence_ids[0]),
    max=int(valid_sentence_ids[-1]),
    step=1,
    description="Sentence ID:",
    layout={"width": "700px"},
)

sentence_id_input = widgets.BoundedIntText(
    value=int(default_sentence_id),
    min=int(valid_sentence_ids[0]),
    max=int(valid_sentence_ids[-1]),
    step=1,
    description="Go to ID:",
    layout={"width": "260px"},
)

min_sim_val = widgets.FloatSlider(
    value=0.80,
    min=0.50,
    max=0.99,
    step=0.01,
    description="Min Sim:",
    layout={"width": "400px"},
)

widgets.jslink((sentence_id_slider, "value"), (sentence_id_input, "value"))

timeline_out = widgets.Output()

def _sentence_id_to_row_index(sentence_id):
    if "node_id" in nodes_df.columns:
        matches = nodes_df.index[nodes_df["node_id"].astype(int) == int(sentence_id)]
        return int(matches[0]) if len(matches) else None

    if 0 <= int(sentence_id) < len(nodes_df):
        return int(sentence_id)
    return None

def show_timeline(sentence_id, min_sim):
    with timeline_out:
        clear_output(wait=True)

        sent_idx = _sentence_id_to_row_index(sentence_id)
        if sent_idx is None:
            print(f"Sentence ID {sentence_id} was not found.")
            return

        row = nodes_df.iloc[sent_idx]

        infl_score = np.nan
        if "node_id" in influence_scores.columns and "influence_score" in influence_scores.columns:
            match = influence_scores.loc[
                influence_scores["node_id"].astype(int) == int(sentence_id),
                "influence_score",
            ]
            if len(match):
                infl_score = float(match.iloc[0])

        score_text = f"{infl_score:.2f}" if not np.isnan(infl_score) else "N/A"

        print(f"\n{'=' * 100}")
        print(
            f"ROOT SENTENCE ID: {int(sentence_id)} | {row['platform'].upper()} ({int(row['year'])}) | Influence Score: {score_text}"
        )
        print(f"{'=' * 100}")
        print(f"{row['sentence'][:400]}{'...' if len(row['sentence']) > 400 else ''}\n")

        fig, res = visualize_sentence_similarity_timeline(
            sent_idx, nodes_df, emb_norm, min_similarity=min_sim
        )
        if fig is not None:
            print(f"Found {len(res) - 1} similar sentences with similarity >= {min_sim:.2f}\n")
            display(fig)
        else:
            print(f"No similar sentences found with similarity >= {min_sim:.2f}\n")

def _on_control_change(_):
    show_timeline(sentence_id_slider.value, min_sim_val.value)

sentence_id_slider.observe(_on_control_change, names="value")
min_sim_val.observe(_on_control_change, names="value")

controls = widgets.VBox([
    widgets.HBox([sentence_id_slider, sentence_id_input]),
    min_sim_val,
])

display(controls)
display(timeline_out)
show_timeline(default_sentence_id, min_sim_val.value)